# Publication figures

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgrfhyL/audio_model_initial_testing/blob/main/Figures.ipynb)

The experiment notebooks write screen figures: PNG at 160 dpi, sized for a notebook cell. That is
the wrong artefact for a paper — ACL templates want **vector PDF**, sized to the column it will sit
in, with type large enough to read at print size.

This notebook redraws every figure from the **stored results**. No GPU, no TIMIT, no model
downloads, no decoding. It reads four numbers-only files that each sweep already writes to Drive
and emits PDF (for LaTeX) alongside PNG (for slides and quick viewing).

| source | file | carries |
|---|---|---|
| **O** scaling | `scaling_per_condition.csv` | pooled S/D/I per (model, offset, arm) |
| **A** delta sweep | `delta_per_utterance.csv` | per-utterance `delta_m` and its four component WERs |
| **B** positional embedding | `pe_per_condition.csv` | corpus WER per (model, condition, arm) |
| **C** localization | `expc_per_utterance.csv` | per-utterance NLL, margins, S/D/I, lengths |

Any file that is missing is skipped with a notice, so this runs usefully even when only some
sweeps have completed.

**Two details that matter for submission.** `pdf.fonttype = 42` embeds TrueType rather than
matplotlib's default Type 3, which some venues and arXiv's checker reject. And figure width is set
to the actual printed column width, so the type is scaled once here rather than by
`\includegraphics` later — shrinking a wide figure in LaTeX is what makes axis labels unreadable.

## 1. Where the data is, and where figures go

In [ ]:
import os, csv, json, math
import numpy as np

try:                                        # Colab: results live on Drive
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/NAACL"
except Exception:                           # local: point this at a folder of downloaded CSVs
    DATA_DIR = os.environ.get("NAACL_DATA", ".")

OUT_DIR = "figures"
os.makedirs(OUT_DIR, exist_ok=True)

SRC = {
    "O": os.path.join(DATA_DIR, "scaling_per_condition.csv"),
    "A": os.path.join(DATA_DIR, "delta_per_utterance.csv"),
    "B": os.path.join(DATA_DIR, "pe_per_condition.csv"),
    "C": os.path.join(DATA_DIR, "expc_per_utterance.csv"),
}
ORDER = ["tiny", "base", "small", "medium", "large-v3"]


def load(key):
    p = SRC[key]
    if not os.path.exists(p):
        print(f"  {key}: MISSING  {p}")
        return None
    rows = list(csv.DictReader(open(p, newline="")))
    print(f"  {key}: {len(rows):6d} rows  {os.path.basename(p)}")
    return rows

print(f"data dir: {DATA_DIR}")
D = {k: load(k) for k in SRC}


def models_in(rows):
    got = {r["model"] for r in rows}
    return [m for m in ORDER if m in got]


def fnum(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return float("nan")

## 2. Publication style

Sizes are in inches at final print scale: ACL two-column gives **3.15 in** for a single column and
**6.30 in** across both. Set the figure to the size it will actually occupy and never rescale it in
LaTeX.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

COL, FULL = 3.15, 6.30                       # ACL single / double column width, inches

mpl.rcParams.update({
    "pdf.fonttype": 42, "ps.fonttype": 42,   # TrueType, not Type 3 (arXiv rejects Type 3)
    "svg.fonttype": "none",
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8.5,
    "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "legend.fontsize": 7,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 3, "ytick.major.size": 3,
    "lines.linewidth": 1.4, "lines.markersize": 4,
    "grid.linewidth": 0.5, "grid.color": "#d9d9d6",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#9a9a96", "axes.labelcolor": "#111111",
    "xtick.color": "#444444", "ytick.color": "#444444",
    "legend.frameon": False, "figure.dpi": 200,
    "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "savefig.facecolor": "white",
})

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
CMODEL = {m: PALETTE[i] for i, m in enumerate(ORDER)}
WRITTEN = []


def finish(fig, stem):
    """Write vector PDF for LaTeX and PNG for everything else."""
    for ext in ("pdf", "png"):
        p = os.path.join(OUT_DIR, f"{stem}.{ext}")
        fig.savefig(p, format=ext)
        WRITTEN.append(p)
    plt.close(fig)
    print(f"  wrote {stem}.pdf + .png")


def grid(ax, axis="y"):
    ax.grid(True, axis=axis, zorder=0)
    ax.set_axisbelow(True)

print("style set | single column", COL, "in | full width", FULL, "in")

## 3. Figure 1 — WER against position, by model size

The paper's first figure. Two panels sharing a y-axis: with timestamps on, and with them off. One
line per model. The y-axis is logarithmic because the series span two orders of magnitude, from
`large-v3` near 0.015 to `tiny` above 1.0 — on a linear axis every model but `tiny` collapses onto
the floor and the flat off-panel becomes unreadable.

The flat right-hand panel is the load-bearing one: same audio, same encoder, same position, only
the decoding objective differs.

In [ ]:
if D["O"]:
    rows = D["O"]
    ms = models_in(rows)
    offs = sorted({int(r["offset_s"]) for r in rows})
    W = {(r["model"], r["timestamps"], int(r["offset_s"])): fnum(r["corpus_wer"]) for r in rows}

    fig, axes = plt.subplots(1, 2, figsize=(FULL, 2.5), sharey=True)
    for ax, ts, title in zip(axes, ("on", "off"),
                             ("timestamps on", "timestamps off")):
        grid(ax)
        for m in ms:
            y = [W.get((m, ts, o), float("nan")) for o in offs]
            ax.plot(offs, y, marker="o", color=CMODEL[m], label=m,
                    markeredgecolor="white", markeredgewidth=0.6, zorder=3)
        ax.set_yscale("log")
        ax.set_xticks(offs)
        ax.set_xlabel("utterance offset in the 30 s window (s)")
        ax.set_title(title, loc="left", fontweight="bold")
    axes[0].set_ylabel("corpus WER")
    axes[1].legend(loc="upper left", ncol=1, handlelength=1.6)
    fig.subplots_adjust(wspace=0.08)
    finish(fig, "fig1_wer_vs_offset")
else:
    print("skipped: run the scaling sweep to produce scaling_per_condition.csv")

## 4. Figure 2 — which quantity tracks the penalty?

The paper's second figure, and the one that settles Experiment C. Three quantities, each divided by
its own value at `tiny`, so three different units become one dimensionless axis and the *shapes*
can be compared:

- **WER penalty** — WER(25 s) / WER(5 s), minus one so a penalty-free model sits at zero
- **ΔNLL** — teacher-forced, paired per utterance, with search removed
- **runaway rate** — fraction of outputs exceeding twice the reference length

A line that tracks the penalty supports the encoder account; one that stays flat while the penalty
falls supports the decoder account.

In [ ]:
def expc_summary(rows):
    """Corpus WER, dNLL and runaway rate per model, from per-utterance rows."""
    by = {}
    for r in rows:
        by.setdefault((r["model"], r["cond"], r["timestamps"]), []).append(r)
    out = {}
    for m in models_in(rows):
        def wer(cond, ts):
            v = by[(m, cond, ts)]
            return (sum(int(x["sub"]) + int(x["dele"]) + int(x["ins"]) for x in v)
                    / sum(int(x["n_ref_words"]) for x in v))
        nll = {c: {x["path"]: fnum(x["nll_text"]) for x in by[(m, c, "on")]}
               for c in ("C0", "C1", "C2")}
        paths = sorted(nll["C0"])
        d1 = np.array([nll["C1"][p] - nll["C0"][p] for p in paths])
        v = by[(m, "C1", "on")]
        run = np.mean([int(x["n_hyp_words"]) > 2 * int(x["n_ref_words"]) for x in v])
        out[m] = {"wer_c0": wer("C0", "on"), "wer_c1": wer("C1", "on"),
                  "dnll": float(d1.mean()), "runaway": float(run),
                  "penalty": wer("C1", "on") / wer("C0", "on")}
    return out


if D["C"]:
    S = expc_summary(D["C"])
    ms = [m for m in ORDER if m in S]
    base = ms[0]
    series = [
        ("WER penalty", [(S[m]["penalty"] - 1) / (S[base]["penalty"] - 1) for m in ms], PALETTE[0]),
        ("$\\Delta$NLL", [S[m]["dnll"] / S[base]["dnll"] for m in ms], PALETTE[1]),
        ("runaway rate", [S[m]["runaway"] / S[base]["runaway"] for m in ms], PALETTE[3]),
    ]
    x = np.arange(len(ms))
    fig, ax = plt.subplots(figsize=(COL, 2.35))
    grid(ax)
    ax.axhline(1.0, color="#9a9a96", linestyle=":", linewidth=0.7, zorder=1)
    for lab, y, c in series:
        ax.plot(x, y, marker="o", color=c, label=lab,
                markeredgecolor="white", markeredgewidth=0.6, zorder=3)
    ax.set_xticks(x); ax.set_xticklabels(ms, rotation=20, ha="right")
    ax.set_ylabel(f"relative to {base}")
    ax.legend(handlelength=1.6)
    finish(fig, "fig2_accounts")

    print(f"\n{'model':>9} {'penalty':>8} {'dNLL':>9} {'runaway':>8}")
    for m in ms:
        print(f"{m:>9} {S[m]['penalty']:7.2f}x {S[m]['dnll']:+9.4f} {S[m]['runaway']:8.3f}")
    pen = np.array([(S[m]["penalty"] - 1) for m in ms])
    print(f"\ncorrelation with the WER penalty across {len(ms)} sizes:")
    print(f"  dNLL     r = {np.corrcoef(pen, [S[m]['dnll'] for m in ms])[0,1]:+.3f}")
    print(f"  runaway  r = {np.corrcoef(pen, [S[m]['runaway'] for m in ms])[0,1]:+.3f}")
else:
    print("skipped: expc_per_utterance.csv not found")

## 5. Figure 3 — the positional-embedding intervention

Experiment B as grouped bars, one group per condition, one bar per model, log y. Reads
`pe_per_condition.csv` directly: it already stores corpus WER per condition, so nothing is
recomputed here.

In [ ]:
if D["B"]:
    rows = [r for r in D["B"] if r["timestamps"] == "on"]
    ms = models_in(rows)
    conds = []
    for r in rows:
        if r["cond"] not in conds:
            conds.append(r["cond"])
    conds.sort()
    W = {(r["model"], r["cond"]): fnum(r["corpus_wer"]) for r in rows}

    x = np.arange(len(conds)); w = 0.8 / len(ms)
    fig, ax = plt.subplots(figsize=(FULL, 2.3))
    grid(ax)
    for i, m in enumerate(ms):
        ax.bar(x + i * w - 0.4 + w / 2, [W.get((m, c), np.nan) for c in conds],
               width=w * 0.9, color=CMODEL[m], label=m, zorder=3, linewidth=0)
    ax.set_yscale("log")
    ax.set_xticks(x); ax.set_xticklabels(conds)
    ax.set_ylabel("corpus WER"); ax.set_xlabel("condition")
    ax.legend(ncol=5, loc="upper center", handlelength=1.2, columnspacing=1.2)
    finish(fig, "fig3_positional_embedding")
else:
    print("skipped: pe_per_condition.csv not found")

## 6. Figure 4 — prevalence and severity of the per-utterance effect

Experiment A. `delta_m` is zero-inflated and heavy-tailed, so prevalence and severity are drawn as
two panels rather than combined into one summary. Left: the share of utterances with
`delta_m > 0`, with Wilson intervals. Right: mean `delta_m`, log scale.

In [ ]:
def wilson(k, n, z=1.959963985):
    if n == 0:
        return float("nan"), 0.0, 1.0
    p, d = k / n, 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, max(0.0, c - h), min(1.0, c + h)


if D["A"]:
    rows = D["A"]
    ms = models_in(rows)
    by = {}
    for r in rows:
        by.setdefault(r["model"], []).append(fnum(r["delta_m"]))

    x = np.arange(len(ms))
    fig, axes = plt.subplots(1, 2, figsize=(FULL, 2.3))

    ax = axes[0]; grid(ax)
    pts = [wilson(int((np.array(by[m]) > 1e-9).sum()), len(by[m])) for m in ms]
    ax.errorbar(x, [p[0] for p in pts],
                yerr=[[p[0] - p[1] for p in pts], [p[2] - p[0] for p in pts]],
                fmt="o", color=PALETTE[0], ecolor="#9a9a96", elinewidth=0.8,
                capsize=2.5, markeredgecolor="white", markeredgewidth=0.6, zorder=3)
    ax.set_xticks(x); ax.set_xticklabels(ms, rotation=20, ha="right")
    ax.set_ylabel(r"$P(\Delta_m > 0)$")
    ax.set_title("prevalence", loc="left", fontweight="bold")

    ax = axes[1]; grid(ax)
    ax.plot(x, [np.mean(by[m]) for m in ms], marker="o", color=PALETTE[1],
            markeredgecolor="white", markeredgewidth=0.6, zorder=3)
    ax.set_yscale("log")
    ax.set_xticks(x); ax.set_xticklabels(ms, rotation=20, ha="right")
    ax.set_ylabel(r"mean $\Delta_m$")
    ax.set_title("severity", loc="left", fontweight="bold")

    fig.subplots_adjust(wspace=0.32)
    finish(fig, "fig4_delta_prevalence_severity")
else:
    print("skipped: delta_per_utterance.csv not found")

## 7. Figure 5 — mean $\Delta_m$ with speaker-clustered intervals

The severity panel of Figure 4 draws the mean with no interval at all. This is the same quantity
as a standalone figure, with the interval the title claims: **BCa bootstrap, resampling the 168
speakers rather than the 1000 utterances.**

### The interval the delta sweep actually drew is not the one its title names

`Colab_DeltaSweep.ipynb` stores both, and they are different keys:

| key in `delta_provenance.json` | resampled unit |
|---|---|
| `mean_ci_lo` / `mean_ci_hi` | utterances |
| `mean_ci_lo_clustered` / `mean_ci_hi_clustered` | speakers |

Its `delta_severity.png` reads `mean_ci_lo`, the **utterance-level** interval, while its title says
"clustered by speaker". The two are numerically close — for `tiny`, `[+1.1998, +1.8453]` against
`[+1.1964, +1.8325]` — so no conclusion moves, but the label is a claim about method and it is
wrong as it stands. This figure recomputes the clustered interval from
`delta_per_utterance.csv`, which is why the file keeps `speaker` alongside `delta_m`.

Nothing is decoded, and no re-run of experiment A is needed: `delta_m` and `speaker` are both in
the numbers-only CSV, and `bootstrap_ci` below is the sweep's implementation verbatim at the same
seed, so the recomputed values reproduce the stored ones exactly. When `delta_provenance.json` is
present the cell asserts that agreement rather than assuming it.

In [ ]:
from statistics import NormalDist

_N = NormalDist()

# --- knobs ---------------------------------------------------------------------------
TITLE  = ("Timestamp-specific positional penalty, mean per utterance\n"
          "(BCa bootstrap 95% CI, clustered by speaker)")
WIDTH  = FULL        # COL gives a single-column version, but the title needs the width
YSCALE = "linear"    # "log" lifts medium/large-v3 off the floor; "linear" keeps the zero line
N_BOOT = 10000       # same as the sweep, so the recomputed interval matches exactly


def bootstrap_ci(x, groups=None, stat=np.mean, n_boot=10000, alpha=0.05, seed=0):
    """BCa bootstrap CI. Verbatim from Colab_DeltaSweep.ipynb section 6.

    groups: cluster labels (speaker). When given, whole clusters are resampled instead of
    individual observations, since utterances from one speaker are not independent.
    """
    x = np.asarray(x, float)
    theta = float(stat(x))
    rng = np.random.default_rng(seed)

    if groups is None:
        n = len(x)
        pool = [np.array([i]) for i in range(n)]
        idx = rng.integers(0, n, (n_boot, n))
        try:
            boots = np.asarray(stat(x[idx], axis=1), dtype=float)
        except TypeError:
            boots = np.array([stat(x[row]) for row in idx])
    else:
        g = np.asarray(groups)
        pool = [np.flatnonzero(g == u) for u in np.unique(g)]
        K = len(pool)
        boots = np.empty(n_boot)
        for b in range(n_boot):
            pick = rng.integers(0, K, K)
            boots[b] = stat(np.concatenate([x[pool[i]] for i in pick]))
    K = len(pool)
    boots = np.sort(boots)

    prop = float(np.mean(boots < theta))
    if prop <= 0.0 or prop >= 1.0:
        return theta, float(boots[0]), float(boots[-1]), "percentile(degenerate)"
    z0 = _N.inv_cdf(prop)

    jack = np.empty(K)
    for i in range(K):
        keep = np.concatenate([pool[j] for j in range(K) if j != i])
        jack[i] = stat(x[keep])
    jbar = jack.mean()
    den = 6.0 * (((jbar - jack) ** 2).sum() ** 1.5)
    a = (((jbar - jack) ** 3).sum() / den) if den != 0 else 0.0

    def adj(p):
        z = _N.inv_cdf(p)
        return _N.cdf(z0 + (z0 + z) / (1 - a * (z0 + z)))

    lo = float(np.quantile(boots, min(max(adj(alpha / 2), 0.0), 1.0)))
    hi = float(np.quantile(boots, min(max(adj(1 - alpha / 2), 0.0), 1.0)))
    return theta, lo, hi, "BCa"


if D["A"]:
    rows = D["A"]
    ms = models_in(rows)

    # rows keep the sweep's original order, so d and spk rebuild its arrays exactly
    by, spk = {}, {}
    for r in rows:
        by.setdefault(r["model"], []).append(fnum(r["delta_m"]))
        spk.setdefault(r["model"], []).append(r["speaker"])

    est = {}
    for m in ms:
        est[m] = bootstrap_ci(np.asarray(by[m], float), groups=np.asarray(spk[m]),
                              n_boot=N_BOOT, seed=0)
        t, lo, hi, meth = est[m]
        print(f"  {m:>9} {t:+.4f}  [{lo:+.4f}, {hi:+.4f}]  {meth}"
              f"  ({len(np.unique(spk[m]))} speakers, {len(by[m])} utterances)")

    # cross-check against the sweep's own stored clustered interval, when it is on Drive
    prov_path = os.path.join(DATA_DIR, "delta_provenance.json")
    if os.path.exists(prov_path):
        prov = json.load(open(prov_path)).get("summary", {})
        drift = [(m, abs(est[m][1] - prov[m]["mean_ci_lo_clustered"]),
                     abs(est[m][2] - prov[m]["mean_ci_hi_clustered"]))
                 for m in ms if m in prov and "mean_ci_lo_clustered" in prov[m]]
        worst = max((max(a, b) for _, a, b in drift), default=None)
        assert worst is None or worst < 1e-9, f"recomputed CI differs from provenance: {drift}"
        print(f"\n  matches delta_provenance.json mean_ci_*_clustered "
              f"on {len(drift)} models (max |drift| {worst:.1e})")
    else:
        print("\n  delta_provenance.json not found; recomputed values not cross-checked")

    x = np.arange(len(ms))
    mid = [est[m][0] for m in ms]
    err = [[est[m][0] - est[m][1] for m in ms], [est[m][2] - est[m][0] for m in ms]]

    fig, ax = plt.subplots(figsize=(WIDTH, 2.5))
    grid(ax)
    if YSCALE == "log":
        ax.set_yscale("log")
    else:
        ax.axhline(0, color="#9a9a96", linewidth=0.6, zorder=2)
    ax.plot(x, mid, color=PALETTE[1], zorder=2)
    ax.errorbar(x, mid, yerr=err, fmt="o", color=PALETTE[1], ecolor="#9a9a96",
                elinewidth=0.8, capsize=2.5, markeredgecolor="white",
                markeredgewidth=0.6, zorder=3)
    for xi, m in zip(x, ms):
        ax.annotate(f"{est[m][0]:+.3f}", (xi, est[m][2]), textcoords="offset points",
                    xytext=(0, 5), ha="center", fontsize=6.5, color="#111111")
    ax.set_xticks(x)
    ax.set_xticklabels(ms, rotation=20, ha="right")
    ax.set_xlim(-0.4, len(ms) - 0.6)
    if YSCALE != "log":                      # headroom for the value annotations
        top = max(est[m][2] for m in ms)
        bot = min(0.0, min(est[m][1] for m in ms))
        ax.set_ylim(bot - 0.04 * (top - bot), top + 0.18 * (top - bot))
    ax.set_ylabel(r"mean $\Delta_m$")
    if TITLE:
        ax.set_title(TITLE, loc="left", fontsize=8.5)
    finish(fig, "fig5_delta_severity_clustered")
else:
    print("skipped: delta_per_utterance.csv not found")

## 8. Figure 6 — how many utterances are affected at all

The companion to Figure 5, and the reason severity alone is not reportable. `delta_m` is
zero-inflated: most utterances are untouched, so a mean over all 1000 mixes "the effect is small"
with "the effect is absent on most clips and large on a few". This figure separates them by
splitting the sign of `delta_m` into a stacked count — hurt, unaffected, helped — rather than a
proportion, so the zero mass is visible as area rather than inferred from what is missing.

No interval is drawn here on purpose. The bars are an exact decomposition of a fixed 1000, not an
estimate; Figure 4's left panel already carries $P(\Delta_m > 0)$ with Wilson intervals for the
inferential version. Like Figure 5 this needs only `delta_per_utterance.csv` — the sign of a stored
column — so neither figure requires experiment A to be re-run.

In [ ]:
TITLE_PREV = "How many utterances are affected at all"
WIDTH_PREV = FULL          # COL fits the title but crowds the legend against the bars

C_HURT, C_NONE, C_HELP = PALETTE[1], "#c9c8c3", PALETTE[0]

if D["A"]:
    rows = D["A"]
    ms = models_in(rows)

    by = {}
    for r in rows:
        by.setdefault(r["model"], []).append(fnum(r["delta_m"]))

    split = {}
    for m in ms:
        d = np.asarray(by[m], float)
        split[m] = (int((d > 1e-9).sum()), int((np.abs(d) <= 1e-9).sum()),
                    int((d < -1e-9).sum()))
        assert sum(split[m]) == len(d), (m, split[m], len(d))
        print(f"  {m:>9}  hurt {split[m][0]:>4}   unaffected {split[m][1]:>4}"
              f"   helped {split[m][2]:>4}   of {len(d)}")

    n_tot = max(len(by[m]) for m in ms)
    x = np.arange(len(ms))
    fig, ax = plt.subplots(figsize=(WIDTH_PREV, 2.5))
    ax.grid(True, axis="y", zorder=0)
    ax.set_axisbelow(True)

    bottom = np.zeros(len(ms))
    for j, (colour, lab) in enumerate(
            ((C_HURT, r"$\Delta_m > 0$  timestamps hurt"),
             (C_NONE, r"$\Delta_m = 0$  no differential effect"),
             (C_HELP, r"$\Delta_m < 0$  timestamps helped"))):
        vals = np.array([split[m][j] for m in ms], float)
        ax.bar(x, vals, bottom=bottom, width=0.6, color=colour, label=lab,
               edgecolor="white", linewidth=0.6, zorder=3)
        bottom += vals

    for xi, m in zip(x, ms):
        if split[m][0]:
            ax.annotate(f"{split[m][0]}", (xi, split[m][0] / 2), ha="center", va="center",
                        fontsize=6.5, color="white", zorder=4)

    ax.set_xticks(x)
    ax.set_xticklabels(ms, rotation=20, ha="right")
    ax.set_xlim(-0.6, len(ms) - 0.4)
    ax.set_ylim(0, n_tot)
    ax.set_ylabel(f"utterances (of {n_tot})")
    ax.legend(ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.02),
              handlelength=1.0, columnspacing=1.4)
    if TITLE_PREV:
        ax.set_title(TITLE_PREV, loc="left", fontsize=8.5, pad=22)
    finish(fig, "fig6_delta_prevalence_stacked")
else:
    print("skipped: delta_per_utterance.csv not found")

## 9. LaTeX tables

The same stored numbers as `booktabs` tables, so Tables 1 and 2 do not have to be transcribed from
notebook output by hand. Paste into the paper and adjust the caption.

In [ ]:
def latex(rows, header, body, caption, label):
    out = ["\\begin{table}[t]", "\\centering", "\\small",
           "\\begin{tabular}{" + header[0] + "}", "\\toprule",
           " & ".join(header[1]) + " \\\\", "\\midrule"]
    out += [" & ".join(r) + " \\\\" for r in body]
    out += ["\\bottomrule", "\\end{tabular}",
            f"\\caption{{{caption}}}", f"\\label{{{label}}}", "\\end{table}"]
    return "\n".join(out)


if D["B"]:
    rows = [r for r in D["B"] if r["timestamps"] == "on"]
    ms = models_in(rows)
    conds = sorted({r["cond"] for r in rows})
    W = {(r["model"], r["cond"]): fnum(r["corpus_wer"]) for r in rows}
    body = []
    for m in ms:
        rec = ""
        if "P0" in conds and "P1" in conds and "P2" in conds:
            d = W[(m, "P1")] - W[(m, "P0")]
            rec = f"{(W[(m,'P1')] - W[(m,'P2')]) / d:.2f}" if d else "--"
        body.append([m] + [f"{W.get((m, c), float('nan')):.4f}" for c in conds] + [rec])
    print(latex(rows, ("l" + "r" * (len(conds) + 1),
                       ["Model"] + conds + ["rec.\\ P2"]), body,
                "Corpus WER under positional-embedding displacement, timestamps on, "
                "1000 TIMIT utterances. Recovery is $(P1-P2)/(P1-P0)$.",
                "tab:pe"))
    print()

if D["A"]:
    ms = models_in(D["A"])
    by = {}
    for r in D["A"]:
        by.setdefault(r["model"], []).append(fnum(r["delta_m"]))
    body = []
    for m in ms:
        d = np.array(by[m]); pos, neg = d > 1e-9, d < -1e-9
        p, lo, hi = wilson(int(pos.sum()), len(d))
        body.append([m, f"{d.mean():+.3f}",
                     f"{d[pos | neg].mean():+.2f}" if (pos | neg).any() else "--",
                     f"{p:.3f}", f"[{lo:.3f}, {hi:.3f}]",
                     f"{pos.sum() / max(1, pos.sum() + neg.sum()):.3f}"])
    print(latex(D["A"], ("lrrrrr",
                ["Model", "mean $\\Delta_m$", "mean\\,$|$aff.", "$P(\\Delta_m{>}0)$",
                 "95\\% CI", "$P(+|$aff.$)$"]), body,
                "Per-utterance timestamp-specific positional penalty over 1000 utterances. "
                "Intervals are Wilson score.", "tab:delta"))

## 10. What was written, and proof it is really vector

A PDF that silently rasterized looks identical in the notebook cell and fails only at submission,
so the check has to be real. Two traps make the obvious version useless:

- **`head[:4] == b"%PDF"` proves nothing.** Every PDF starts that way, raster or not.
- **Grepping the file bytes for `/Type3` also proves nothing.** matplotlib Flate-compresses its
  object streams, so `/FontFile2`, `/Type3` and `/Subtype /Image` are all invisible to a byte
  scan — searching raw bytes returns zero hits on a perfectly good file *and* on a rasterized one.

So the streams are inflated first, then scanned. The assertion is that `/Subtype /Image`,
`/DCTDecode` and `/Type3` are all absent, and that `/FontFile2` is present — the positive evidence
that `pdf.fonttype = 42` took effect and the fonts are embedded TrueType subsets rather than
matplotlib's default Type 3.

Everything is then zipped and pulled to local disk with `files.download()`, which reads the
runtime's filesystem. A copy on Drive does not substitute: the runtime is recycled and the figures
go with it.

In [ ]:
import re, shutil, zlib

WRITTEN = list(dict.fromkeys(WRITTEN))          # re-running a cell appends; dedupe
print(f"{len(WRITTEN)} files in {os.path.abspath(OUT_DIR)}/\n")
for p in WRITTEN:
    print(f"  {os.path.getsize(p)/1024:7.1f} KB  {os.path.basename(p)}")


def pdf_blobs(path):
    """File bytes plus every FlateDecode stream that inflates.

    matplotlib compresses its object streams, so /FontFile2 and /Type3 do not appear in the
    raw bytes at all -- a byte-level grep passes on every file and checks nothing.
    """
    raw = open(path, "rb").read()
    out = [raw]
    for m in re.finditer(rb"stream\r?\n", raw):
        s, e = m.end(), raw.find(b"endstream", m.end())
        if e < 0:
            continue
        try:
            out.append(zlib.decompress(raw[s:e]))
        except zlib.error:                      # not Flate, or not a whole stream
            pass
    return b"\n".join(out)


RASTER = (b"/Subtype /Image", b"/DCTDecode", b"/JPXDecode")
pdfs = [p for p in WRITTEN if p.endswith(".pdf")]
for p in pdfs:
    assert open(p, "rb").read(4) == b"%PDF", f"{p} is not a PDF"
    body = pdf_blobs(p)
    bad = [m.decode() for m in RASTER if m in body]
    assert not bad, f"{os.path.basename(p)} contains rasterized content: {bad}"
    assert b"/Type3" not in body, (
        f"{os.path.basename(p)} embeds Type 3 fonts; arXiv rejects these. "
        "Is pdf.fonttype = 42 set before the figure was drawn?")
    assert b"/FontFile2" in body, (
        f"{os.path.basename(p)} embeds no TrueType font; expected /FontFile2 from fonttype 42")
print(f"\nall {len(pdfs)} PDFs verified vector: no /Subtype /Image, no /DCTDecode, no /Type3,"
      f"\nfonts embedded as TrueType subsets (/FontFile2 present in each)")

# --- pull to local disk; a copy on Drive dies with the runtime ------------------------
zip_path = shutil.make_archive(os.path.join(os.getcwd(), "figures"), "zip", OUT_DIR)
print(f"\n{os.path.basename(zip_path)}: {os.path.getsize(zip_path)/1024:.1f} KB "
      f"({len(pdfs)} PDF + {len(WRITTEN)-len(pdfs)} PNG)")
try:
    from google.colab import files
    files.download(zip_path)                    # one prompt for the whole set
    print("download started -- check your browser's downloads")
except Exception as e:                          # not on Colab, or download blocked
    print(f"not downloading ({type(e).__name__}); the zip is on disk at the path above")